# 🕵️ Aula 8 — Testando Positivo (parte 4, final): um detector que mede estresse

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Este é o último notebook do projeto que atravessa as **Aulas 5 a 8**. Até aqui, o detector de mentiras dava um resultado de "sim ou não": passou ou falhou. Agora imaginamos um detector mais realista, que mede um **nível de estresse**, um número que pode ser **qualquer valor** entre 0 e 10 (como 6,37 ou 8,002). Isso é uma **variável aleatória contínua**, o assunto da Aula 8.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

E guarda os números do problema: o nível de estresse vai de **0 a 10**, e o detector **sinaliza como suspeito** quem passar de **7**.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a conta da aula diz.

In [ ]:
import numpy as np                  # calculadora de números e sorteios
import matplotlib.pyplot as plt     # gráficos

a, b = 0, 10        # o nível de estresse X vai de 0 a 10
limiar = 7          # acima de 7, o funcionário é sinalizado como suspeito
print("Tudo pronto!")

## 1. A densidade: como a probabilidade se espalha

### A situação
O novo detector mede o nível de estresse **X** de cada funcionário durante a entrevista. Vamos supor o caso mais simples: **todos os valores entre 0 e 10 são igualmente prováveis**. O funcionário é sinalizado se **X > 7**.

### A pergunta
Como representar "todos os valores igualmente prováveis", se existem **infinitos** valores entre 0 e 10?

### A ferramenta da aula
A **função densidade de probabilidade** $f(x)$. Numa variável contínua, a probabilidade não fica "em cima" de cada ponto: ela é a **área embaixo da curva** da densidade. Por isso a área total embaixo da curva tem que ser **1**.

Se todos os valores são igualmente prováveis, a densidade é **constante** (um retângulo) entre 0 e 10. Para a área do retângulo dar 1, a altura tem que ser:

$$f(x) = \frac{1}{10} = 0{,}1 \quad \text{para } 0 \le x \le 10$$

### O que o código faz
Desenha essa densidade (uma linha reta na altura 0,1) e **pinta a área** que corresponde a "X > 7", ou seja, os funcionários sinalizados.

In [ ]:
x = np.linspace(a, b, 200)                      # 200 pontos entre 0 e 10, para desenhar a curva
densidade = np.full_like(x, 1 / (b - a))        # altura constante: 1/10

plt.plot(x, densidade, color="steelblue")
plt.fill_between(x[x > limiar], densidade[x > limiar], color="coral", alpha=0.6, label="X > 7 (sinalizado)")
plt.ylim(0, 0.15)
plt.xlabel("Nível de estresse (X)")
plt.ylabel("Densidade f(x)")
plt.title("Densidade de X, com a área dos sinalizados em destaque")
plt.legend()
plt.show()

**Como ler:** a linha azul é a densidade, um retângulo de base 10 e altura 0,1 (área total 1). A **área laranja** é a probabilidade de um funcionário ser sinalizado. Repare que o eixo vertical **não é probabilidade**: é densidade. A probabilidade é sempre uma **área**.

## 2. Que fração dos funcionários é sinalizada?

### A pergunta
Quanto vale $P(X > 7)$?

### A ferramenta da aula
A probabilidade de um intervalo é a **área embaixo da densidade** nesse intervalo (a integral de $f(x)$). Como aqui a densidade é constante, a área é só a de um **retângulo**: base × altura.

### A resposta pela conta
$$P(X > 7) = \underbrace{(10 - 7)}_{\text{base}} \times \underbrace{0{,}1}_{\text{altura}} = 0{,}3$$

Ou seja, **30%** dos funcionários seriam sinalizados.

### O que o código faz
Primeiro, faz a mesma conta (base × altura). Depois, **simula 500.000 funcionários**: o sorteador dá a cada um um nível de estresse qualquer entre 0 e 10 (todos igualmente prováveis, com `uniform`) e conta que fração passou de 7. A **semente 3** faz o sorteio sair sempre igual.

In [ ]:
p_sinalizado = (b - limiar) * (1 / (b - a))                 # área do retângulo: base × altura
print(f"Teórico:  P(X > 7) = (10 - 7) × 0.1 = {p_sinalizado:.3f}")

rng = np.random.default_rng(3)                              # sorteador, semente 3
n_funcionarios = 500_000
estresse = rng.uniform(a, b, size=n_funcionarios)           # um nível de estresse para cada funcionário

print(f"Empírico: fração com X > 7 = {(estresse > limiar).mean():.3f}")

**Como ler:** a conta dá **0,300** e a simulação dá **0,301**. A área do retângulo previu corretamente a fração de sinalizados.

## 3. E a chance de o estresse dar **exatamente** 7?

### A pergunta
Qual a probabilidade de um funcionário ter nível de estresse **exatamente igual a 7** (7,000000...)? Essa pergunta confundiu muita gente na verificação do início da aula.

### A ferramenta da aula
Probabilidade = área. Um **único ponto** é um retângulo de **base zero**, então tem área zero:

$$P(X = 7) = 0 \times 0{,}1 = 0$$

Isso vale para **qualquer** variável contínua e **qualquer** valor, e não depende da distribuição. Por isso, em variáveis contínuas, $P(X > 7) = P(X \ge 7)$.

### O que o código faz
Usa os 500.000 níveis de estresse sorteados na seção anterior e conta:
1. quantos deram **exatamente** 7;
2. quantos caíram numa **janela em volta de 7**, cada vez mais estreita (largura 1; 0,1; 0,01; 0,001).

In [ ]:
print(f"Exatamente iguais a 7: {(estresse == 7).sum()} de {n_funcionarios}")
print()
for largura in [1, 0.1, 0.01, 0.001]:
    na_janela = (estresse > 7 - largura / 2) & (estresse < 7 + largura / 2)   # quem ficou perto de 7
    print(f"Janela de largura {largura:<6}: fração {na_janela.mean():.5f}   (área teórica: {largura * 0.1:.5f})")

**Como ler:** **nenhum** dos 500.000 funcionários teve estresse exatamente 7. E a fração dentro da janela **cai junto com a largura**: cerca de 0,1 para largura 1, 0,01 para 0,1, e assim por diante, sempre perto de "largura × 0,1". Quando a janela vira um ponto (largura 0), a área, e portanto a probabilidade, vira **zero**.

## 4. Esperança e variância do estresse

### A pergunta
Qual o nível de estresse **médio** dos funcionários, e quanto ele **varia**?

### A ferramenta da aula
Para variáveis contínuas, as somas da Aula 7 viram **integrais**:

$$E[X] = \int x\, f(x)\, dx \qquad Var[X] = \int (x - E[X])^2 f(x)\, dx$$

### A resposta pela conta
Com $f(x) = 0{,}1$ entre 0 e 10:

$$E[X] = \int_0^{10} x \cdot 0{,}1\, dx = 0{,}1 \cdot \frac{10^2}{2} = 5 \qquad Var[X] = \int_0^{10} (x-5)^2 \cdot 0{,}1\, dx = \frac{100}{12} \approx 8{,}333$$

### O que o código faz
Calcula a **média** e a **variância** dos 500.000 níveis de estresse simulados e compara com os valores das integrais.

In [ ]:
E_teorico = 5
Var_teorico = 100 / 12

print(f"E[X]   teórico: {E_teorico:.3f}   |   empírico: {estresse.mean():.3f}")
print(f"Var[X] teórico: {Var_teorico:.3f}   |   empírico: {estresse.var():.3f}")

**Como ler:** a média simulada fica em **5,008** e a variância em **8,331**, bem perto de 5 e 8,333. Faz sentido a média ser o **meio do intervalo**: como nenhum valor é mais provável que outro, o "centro de gravidade" da distribuição é o ponto médio.

> Essa distribuição, com densidade constante num intervalo, tem nome: **uniforme contínua**. Ela volta com esse nome, junto com outras, na **Aula 12**.

## 5. Fechando o projeto: as quatro partes juntas

A mesma pergunta, **"quão confiável é um teste de triagem?"**, foi respondida com quatro ferramentas diferentes:

| Aula | Ferramenta | O que descobrimos |
|---|---|---|
| 5 | Probabilidade condicional (simulação) | Entre os que falham no detector, só **cerca de 31%** são ladrões |
| 6 | Probabilidade Total e Bayes | $P(\text{falhou}) = 0{,}26$ e $P(\text{ladrão} \mid \text{falhou}) = 0{,}08/0{,}26 \approx 0{,}31$, agora pela fórmula |
| 7 | Variável aleatória discreta | Numa equipe de 3, $E[X] = 0{,}78$ falhas, e há cerca de 59% de chance de pelo menos uma |
| 8 | Variável aleatória contínua | Com o detector de estresse, $P(X > 7) = 0{,}3$, $E[X] = 5$ e $P(X = 7) = 0$ |

A lição geral: o **mesmo problema real** pode ser atacado de vários jeitos, e a ferramenta depende do **tipo de dado** que você tem: um "sim/não", uma contagem ou uma medida contínua.

Na **Aula 9** começamos os **modelos de distribuição**: famílias prontas (uniforme discreta, Bernoulli, geométrica...) que resolvem de uma vez situações que aparecem o tempo todo.